# Sprint 2b.4 — Interpretable Regression Model
## AI for Equitable Public Transportation | Deloitte Capstone

**Purpose:** Build an interpretable regression model that explains *which factors drive transit equity gaps* in Miami-Dade County census tracts.

**Input:** `Sprint2b_Modeling_Features_NotebookOutput.csv` (504 tracts × 67 columns) from Luna's feature engineering notebook.

**Key output:**
- Coefficient table: "a 1-unit increase in X changes equity score by Y"
- Feature importance ranking via Lasso selection
- Cross-validated performance metrics (R², RMSE, MAE)
- Residual diagnostics

| Version | Date | Changes |
|---------|------|---------|
| v1 | 2026-03-16 | Initial build: target leakage audit, Lasso selection, Ridge regression, 5-fold CV |

## 1. Setup and Imports

Load all required libraries. We use scikit-learn for modeling, with Lasso for feature selection and Ridge for stable coefficient estimation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LassoCV, RidgeCV, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 70)
pd.set_option('display.float_format', '{:.4f}'.format)
np.random.seed(42)

print("Libraries loaded successfully.")

## 2. Load Data

Load the modeling-ready dataset produced by Sprint 2b feature engineering. This CSV has 504 census tracts with 67 columns covering demographics, GTFS transit metrics, ACS temporal trends, spatial features, and interaction terms.

In [ ]:
df = pd.read_csv('Sprint2b_Modeling_Features_NotebookOutput.csv', dtype={'tract_geoid': str})
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

## 3. Target Leakage Audit

**This is the most critical step in the entire notebook.**

Our target variable is `equity_priority_score`. Luna computed it using this formula:

```
equity_priority_score = composite_need × composite_access_deficit

WHERE:
  composite_need = (ind_1_transit_dependency + ind_6_economic_vulnerability) / 2
  composite_access_deficit = (ind_2 + ind_3 + ind_4 + ind_5 + ind_7) / 5
```

**Target leakage** means including features that are derived from the target or that directly encode the target's value. If we leave these in, the model will simply reverse-engineer Luna's formula instead of learning genuine relationships in the data.

### Columns that MUST be excluded:

| Column | Reason |
|--------|--------|
| `equity_priority_score` | This IS the target — not a feature |
| `equity_tier` | Percentile-based label derived from the target |
| `equity_percentile` | Rank transform of the target (r=0.936) |
| `composite_need` | Direct multiplicative component of the target |
| `composite_access_deficit` | Direct multiplicative component of the target |
| `ind_1_transit_dependency` | Component of composite_need |
| `ind_2_temporal_mismatch` | Component of composite_access_deficit |
| `ind_3_structural_gap` | Component of composite_access_deficit |
| `ind_4_time_tax` | Component of composite_access_deficit |
| `ind_5_service_coverage` | Component of composite_access_deficit |
| `ind_6_economic_vulnerability` | Component of composite_need |
| `ind_7_multimodal_deficit` | Component of composite_access_deficit |
| `institutional_tract` | Zero variance (all False) — no information |
| `neighbor_mean_equity_score` | Derived from neighbors' equity_priority_score — indirect leakage |
| `tract_geoid` | Identifier, not a feature |

After removing these 15 columns, we retain only **raw, independently-measured features** that the model must learn to combine.

In [ ]:
# Define target
TARGET = 'equity_priority_score'

# Columns to EXCLUDE — target leakage, identifiers, zero-variance
LEAKAGE_COLS = [
    # Target and direct derivatives
    'equity_priority_score',
    'equity_tier',
    'equity_percentile',
    # Composite components (these ARE the target formula)
    'composite_need',
    'composite_access_deficit',
    # Individual indicators (components of composites)
    'ind_1_transit_dependency',
    'ind_2_temporal_mismatch',
    'ind_3_structural_gap',
    'ind_4_time_tax',
    'ind_5_service_coverage',
    'ind_6_economic_vulnerability',
    'ind_7_multimodal_deficit',
    # Indirect leakage
    'neighbor_mean_equity_score',  # computed from neighbors' equity_priority_score
    # Zero-variance / identifiers
    'institutional_tract',         # all False
    'tract_geoid',                 # identifier
]

# Separate target and features
y = df[TARGET].copy()
feature_cols = [c for c in df.columns if c not in LEAKAGE_COLS]
X = df[feature_cols].copy()

print(f"Target: {TARGET}")
print(f"  shape: {y.shape}, min={y.min():.4f}, max={y.max():.4f}, mean={y.mean():.4f}")
print(f"\nExcluded {len(LEAKAGE_COLS)} leakage/identifier columns:")
for col in LEAKAGE_COLS:
    if col in df.columns:
        print(f"  ✗ {col}")
    else:
        print(f"  ✗ {col} (not found — OK)")

print(f"\nRetained {X.shape[1]} features:")
for col in X.columns:
    print(f"  ✓ {col}")

### 3.1 Leakage Verification

As a safety check, we verify that none of the retained features have suspiciously high correlation with the target. In a clean feature set, no single raw feature should have |r| > 0.85 with the target (that would suggest it's a near-duplicate of the target formula).

If any feature exceeds this threshold, we investigate before proceeding.

In [ ]:
# Compute correlations between retained features and target
feature_target_corr = X.corrwith(y).abs().sort_values(ascending=False)

print("Feature-target correlations (|r|, descending):")
print("=" * 60)
for col in feature_target_corr.index:
    r = X[col].corr(y)
    flag = " ⚠ INVESTIGATE" if abs(r) > 0.85 else ""
    print(f"  {col:45s}  r = {r:+.4f}{flag}")

max_corr = feature_target_corr.max()
print(f"\nMax |r| with target: {max_corr:.4f}")
if max_corr > 0.85:
    print("⚠ WARNING: Some features have high target correlation — review above.")
else:
    print("✓ PASS: No feature has suspicious correlation with target.")

## 4. Feature Diagnostics

Before modeling, we check for issues that would compromise the regression:
1. **Missing values** — any NaN remaining?
2. **Zero-variance features** — features with no variation carry no information
3. **Multicollinearity** — highly correlated feature pairs destabilize regression coefficients

We address each issue before fitting any model.

In [ ]:
# Check 1: Missing values
nan_counts = X.isnull().sum()
total_nan = nan_counts.sum()
print(f"[1] Missing values: {total_nan}")
if total_nan > 0:
    print(nan_counts[nan_counts > 0])
else:
    print("    ✓ No missing values")

# Check 2: Zero or near-zero variance
low_var = X.columns[X.std() < 1e-6].tolist()
print(f"\n[2] Zero-variance features: {len(low_var)}")
if low_var:
    print(f"    Dropping: {low_var}")
    X = X.drop(columns=low_var)
else:
    print("    ✓ All features have meaningful variance")

# Check 3: Multicollinearity (|r| > 0.90 between features)
corr_matrix = X.corr()
high_corr_pairs = []
for i in range(len(corr_matrix)):
    for j in range(i + 1, len(corr_matrix)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.90:
            high_corr_pairs.append((corr_matrix.index[i], corr_matrix.columns[j], r))

print(f"\n[3] Feature pairs with |r| > 0.90: {len(high_corr_pairs)}")
if high_corr_pairs:
    for c1, c2, r in sorted(high_corr_pairs, key=lambda x: -abs(x[2])):
        print(f"    {c1:40s} × {c2:40s}  r={r:+.3f}")
else:
    print("    ✓ No highly correlated pairs")

### 4.1 Handle Multicollinearity

For pairs with |r| > 0.90, we drop the feature that has **lower correlation with the target**. This keeps the more informative feature while removing redundancy.

Lasso regression can handle some collinearity on its own, but for the interpretable Ridge model, removing near-duplicates ensures stable, meaningful coefficients.

In [ ]:
# Drop the weaker feature from each highly correlated pair
drop_cols = set()
for c1, c2, r in high_corr_pairs:
    if c1 in drop_cols or c2 in drop_cols:
        continue  # already handled
    # Keep the one with higher |correlation| to target
    r1 = abs(X[c1].corr(y))
    r2 = abs(X[c2].corr(y))
    drop = c2 if r1 >= r2 else c1
    keep = c1 if r1 >= r2 else c2
    drop_cols.add(drop)
    print(f"  Pair: {c1} × {c2} (r={r:+.3f})")
    print(f"    Keep: {keep} (|r_target|={max(r1,r2):.3f})")
    print(f"    Drop: {drop} (|r_target|={min(r1,r2):.3f})")
    print()

if drop_cols:
    X = X.drop(columns=list(drop_cols))
    print(f"Dropped {len(drop_cols)} redundant features.")
else:
    print("No features to drop.")

print(f"\nFinal feature set: {X.shape[1]} features × {X.shape[0]} tracts")

## 5. Target Distribution Analysis

The equity_priority_score is computed as `composite_need × composite_access_deficit`, which means it is the product of two [0,1] values. This creates a right-skewed distribution (most values are low, with a long right tail).

Linear regression assumes approximately normally-distributed residuals. A log-transform of the target helps satisfy this assumption and stabilizes the variance of errors across the prediction range.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Original distribution
axes[0].hist(y, bins=40, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_title(f'Original: equity_priority_score\nskew={y.skew():.2f}, kurtosis={y.kurtosis():.2f}')
axes[0].set_xlabel('equity_priority_score')
axes[0].set_ylabel('Count')
axes[0].axvline(y.median(), color='red', linestyle='--', label=f'median={y.median():.3f}')
axes[0].legend()

# Log-transformed distribution
y_log = np.log1p(y)  # log(1 + x) to handle values near 0
axes[1].hist(y_log, bins=40, edgecolor='black', alpha=0.7, color='coral')
axes[1].set_title(f'Log-transformed: log(1 + score)\nskew={y_log.skew():.2f}, kurtosis={y_log.kurtosis():.2f}')
axes[1].set_xlabel('log(1 + equity_priority_score)')
axes[1].set_ylabel('Count')
axes[1].axvline(y_log.median(), color='red', linestyle='--', label=f'median={y_log.median():.3f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Log transform reduces skew from {y.skew():.2f} to {y_log.skew():.2f}")
print("Using log-transformed target for regression.")

## 6. Feature Standardization

We standardize all features to zero mean and unit variance before regression. This is necessary because:

1. **Features are on different scales:** Trend slopes range from -2,700 to +24,000 while percentages range from 0 to 100 and ratios from 0 to 1. Without standardization, the model would favor large-scale features.
2. **Coefficient comparability:** After standardization, each coefficient represents the effect of a 1-standard-deviation change, making coefficients directly comparable across features.
3. **Lasso penalization fairness:** Lasso penalizes coefficient magnitude. Unscaled features with naturally larger values would be unfairly penalized.

In [ ]:
# Standardize features (z-score: mean=0, std=1)
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)

# Use log-transformed target
y_model = np.log1p(y)

print(f"Features standardized: {X_scaled.shape}")
print(f"Target: log(1 + equity_priority_score)")
print(f"\nFeature means (should be ~0):")
print(X_scaled.mean().describe())
print(f"\nFeature stds (should be ~1):")
print(X_scaled.std().describe())

## 7. Feature Selection via Lasso Regression

Lasso (L1 regularization) drives irrelevant feature coefficients exactly to zero. We use `LassoCV` with 5-fold cross-validation to automatically find the optimal regularization strength (alpha).

Features that survive with non-zero coefficients are the ones Lasso considers most important for predicting equity scores. This serves as an automatic feature filter before we fit the final interpretable model.

We use stratified folds based on `equity_tier` so that each fold has a proportional representation of Critical, High, Moderate, and Low tracts.

In [ ]:
# Create stratification labels for CV
# (use equity_tier from original dataframe for stratification only — not as a feature)
strat_labels = df['equity_tier'].values

# Stratified 5-fold CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# LassoCV: test 100 alpha values, pick the one with best CV score
lasso_cv = LassoCV(
    n_alphas=100,
    cv=cv.split(X_scaled, strat_labels),
    max_iter=10000,
    random_state=42
)
lasso_cv.fit(X_scaled, y_model)

print(f"Optimal alpha: {lasso_cv.alpha_:.6f}")
print(f"R² at optimal alpha: {lasso_cv.score(X_scaled, y_model):.4f}")

# Extract surviving features (non-zero coefficients)
lasso_coefs = pd.Series(lasso_cv.coef_, index=X_scaled.columns)
surviving = lasso_coefs[lasso_coefs != 0].abs().sort_values(ascending=False)
eliminated = lasso_coefs[lasso_coefs == 0]

print(f"\nFeatures surviving Lasso: {len(surviving)} / {len(lasso_coefs)}")
print(f"Features eliminated (coef = 0): {len(eliminated)}")

print(f"\n{'='*60}")
print("SURVIVING FEATURES (sorted by |coefficient|):")
print(f"{'='*60}")
for feat in surviving.index:
    coef = lasso_coefs[feat]
    print(f"  {feat:45s}  coef = {coef:+.6f}")

if len(eliminated) > 0:
    print(f"\nEliminated features:")
    for feat in eliminated.index:
        print(f"  ✗ {feat}")

## 8. Ridge Regression on Selected Features

Now we fit a Ridge regression (L2 regularization) using only the features that survived Lasso selection. Why Ridge instead of using Lasso's coefficients directly?

1. **Lasso shrinks coefficients toward zero aggressively** — the surviving coefficients are biased downward. Ridge gives more accurate coefficient magnitudes.
2. **Ridge handles residual multicollinearity** among the surviving features better than OLS.
3. **The coefficients from Ridge are our interpretable output** — "a 1-std increase in feature X is associated with a Y change in log equity score."

We use `RidgeCV` to find the optimal alpha, then extract the final coefficient table.

In [ ]:
# Select only Lasso-surviving features
selected_features = surviving.index.tolist()
X_selected = X_scaled[selected_features]

print(f"Fitting Ridge regression on {len(selected_features)} features...")

# RidgeCV with built-in cross-validation
ridge_cv = RidgeCV(
    alphas=np.logspace(-3, 3, 100),
    cv=cv.split(X_selected, strat_labels),
    scoring='r2'
)
ridge_cv.fit(X_selected, y_model)

print(f"Optimal alpha: {ridge_cv.alpha_:.4f}")
print(f"R² (training): {ridge_cv.score(X_selected, y_model):.4f}")

# Coefficient table
coef_table = pd.DataFrame({
    'feature': selected_features,
    'ridge_coefficient': ridge_cv.coef_,
    'abs_coefficient': np.abs(ridge_cv.coef_),
}).sort_values('abs_coefficient', ascending=False)

print(f"\n{'='*70}")
print("RIDGE REGRESSION COEFFICIENTS")
print(f"{'='*70}")
print(f"{'Feature':45s} {'Coefficient':>12s} {'Direction':>10s}")
print(f"{'-'*70}")
for _, row in coef_table.iterrows():
    direction = "↑ increases" if row['ridge_coefficient'] > 0 else "↓ decreases"
    print(f"  {row['feature']:45s} {row['ridge_coefficient']:+.6f}   {direction}")
print(f"\nIntercept: {ridge_cv.intercept_:.6f}")
print(f"\nInterpretation: A 1-standard-deviation increase in the feature")
print(f"is associated with the shown change in log(1 + equity_priority_score).")

## 9. Cross-Validation Performance

We evaluate the Ridge model using 5-fold stratified cross-validation. Each fold trains on 80% of the data and tests on the held-out 20%. We report three metrics:

- **R²**: Proportion of variance explained (1.0 = perfect, 0 = baseline)
- **RMSE**: Root mean squared error (same units as target)
- **MAE**: Mean absolute error (robust to outliers)

We compute these in the **original scale** (not log-transformed) so the errors are interpretable as equity score units.

In [ ]:
# Cross-validated predictions (in log space, converted back)
from sklearn.model_selection import cross_val_predict

y_pred_log = cross_val_predict(
    Ridge(alpha=ridge_cv.alpha_),
    X_selected,
    y_model,
    cv=cv.split(X_selected, strat_labels)
)

# Convert predictions back to original scale
y_pred = np.expm1(y_pred_log)  # inverse of log1p
y_true = y.values

# Metrics in original scale
r2 = r2_score(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)

print(f"{'='*50}")
print(f"5-FOLD CROSS-VALIDATION RESULTS (original scale)")
print(f"{'='*50}")
print(f"  R²:   {r2:.4f}")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE:  {mae:.4f}")
print(f"\n  Target range: [{y_true.min():.4f}, {y_true.max():.4f}]")
print(f"  Target std:   {y_true.std():.4f}")
print(f"  MAE / std:    {mae / y_true.std():.3f} (lower = better, <0.5 is decent)")

# Per-fold scores
print(f"\nPer-fold R² scores:")
fold_scores = cross_val_score(
    Ridge(alpha=ridge_cv.alpha_),
    X_selected, y_model,
    cv=cv.split(X_selected, strat_labels),
    scoring='r2'
)
for i, score in enumerate(fold_scores):
    print(f"  Fold {i+1}: R² = {score:.4f}")
print(f"  Mean:   R² = {fold_scores.mean():.4f} ± {fold_scores.std():.4f}")

## 10. Residual Analysis

Residual analysis tells us whether the model's errors are random (good) or systematic (bad). We check:

1. **Residuals vs Predicted**: Should show no pattern. A funnel shape means variance changes with prediction (heteroscedasticity).
2. **Residuals by Equity Tier**: Does the model perform worse for certain tiers? Critical tracts are the most important to get right.
3. **Residual distribution**: Should be approximately normal and centered at zero.

In [ ]:
residuals = y_true - y_pred

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Plot 1: Residuals vs Predicted
axes[0].scatter(y_pred, residuals, alpha=0.4, s=15, color='steelblue')
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('Predicted equity_priority_score')
axes[0].set_ylabel('Residual (true - predicted)')
axes[0].set_title('Residuals vs Predicted')

# Plot 2: Residuals by tier
tier_order = ['Low', 'Moderate', 'High', 'Critical']
tier_colors = {'Low': '#2ecc71', 'Moderate': '#f39c12', 'High': '#e74c3c', 'Critical': '#8e44ad'}
tiers = df['equity_tier'].values

for tier in tier_order:
    mask = tiers == tier
    axes[1].scatter(
        y_pred[mask], residuals[mask],
        alpha=0.5, s=15, label=tier, color=tier_colors[tier]
    )
axes[1].axhline(0, color='red', linestyle='--', linewidth=1)
axes[1].set_xlabel('Predicted equity_priority_score')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residuals by Equity Tier')
axes[1].legend(fontsize=8)

# Plot 3: Residual distribution
axes[2].hist(residuals, bins=40, edgecolor='black', alpha=0.7, color='steelblue')
axes[2].axvline(0, color='red', linestyle='--', linewidth=1)
axes[2].set_xlabel('Residual')
axes[2].set_ylabel('Count')
axes[2].set_title(f'Residual Distribution\nmean={residuals.mean():.4f}, std={residuals.std():.4f}')

plt.tight_layout()
plt.savefig('residual_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Per-tier error breakdown
print(f"\n{'='*60}")
print(f"ERROR BREAKDOWN BY EQUITY TIER")
print(f"{'='*60}")
print(f"{'Tier':12s} {'Count':>6s} {'MAE':>8s} {'RMSE':>8s} {'Mean Resid':>11s}")
print(f"{'-'*60}")
for tier in tier_order:
    mask = tiers == tier
    t_resid = residuals[mask]
    t_mae = np.abs(t_resid).mean()
    t_rmse = np.sqrt((t_resid ** 2).mean())
    print(f"  {tier:10s} {mask.sum():6d} {t_mae:8.4f} {t_rmse:8.4f} {t_resid.mean():+11.4f}")

## 11. Feature Importance Visualization

A horizontal bar chart of the Ridge regression coefficients, sorted by magnitude. Positive coefficients (right) mean the feature *increases* the equity priority score (i.e., worsens equity). Negative coefficients (left) mean the feature *improves* equity.

This is the primary interpretability output — the answer to "what drives transit inequity in Miami-Dade?".

In [ ]:
# Plot coefficients
plot_df = coef_table.sort_values('ridge_coefficient')

fig, ax = plt.subplots(figsize=(10, max(6, len(plot_df) * 0.35)))

colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in plot_df['ridge_coefficient']]
ax.barh(plot_df['feature'], plot_df['ridge_coefficient'], color=colors, edgecolor='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Ridge Coefficient (standardized)')
ax.set_title('Feature Importance — Interpretable Regression Model\n(Red = worsens equity | Green = improves equity)')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('feature_importance_regression.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Actual vs Predicted Scatter

A scatter plot comparing true equity scores to model predictions. Points on the diagonal line represent perfect predictions. Deviation from the diagonal shows prediction error.

This gives an intuitive view of model accuracy across the entire score range.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

for tier in tier_order:
    mask = tiers == tier
    ax.scatter(y_true[mask], y_pred[mask], alpha=0.5, s=20, 
               label=f'{tier} ({mask.sum()})', color=tier_colors[tier])

# Perfect prediction line
lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
ax.plot(lims, lims, 'k--', linewidth=1, label='Perfect prediction')

ax.set_xlabel('True equity_priority_score')
ax.set_ylabel('Predicted equity_priority_score')
ax.set_title(f'Actual vs Predicted (5-fold CV)\nR² = {r2:.4f}, RMSE = {rmse:.4f}')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('actual_vs_predicted_regression.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Save Outputs

Export the coefficient table and cross-validated predictions for use in Notebook 2 (ML model comparison) and Sprint 3 (simulation baseline).

In [ ]:
# Save coefficient table
coef_table.to_csv('Sprint2b_Ridge_Coefficients.csv', index=False)
print(f"Saved: Sprint2b_Ridge_Coefficients.csv ({len(coef_table)} features)")

# Save predictions with tract IDs
predictions_df = pd.DataFrame({
    'tract_geoid': df['tract_geoid'].values,
    'equity_tier': df['equity_tier'].values,
    'true_score': y_true,
    'predicted_score_ridge': y_pred,
    'residual': residuals,
})
predictions_df.to_csv('Sprint2b_Ridge_Predictions.csv', index=False)
print(f"Saved: Sprint2b_Ridge_Predictions.csv ({len(predictions_df)} tracts)")

# Save the list of selected features (for Notebook 2 consistency)
with open('Sprint2b_Selected_Features.txt', 'w') as f:
    for feat in selected_features:
        f.write(feat + '\n')
print(f"Saved: Sprint2b_Selected_Features.txt ({len(selected_features)} features)")

## 14. Summary

### Key Results

| Metric | Value |
|--------|-------|
| Features selected by Lasso | See count above |
| Cross-validated R² | See output above |
| Cross-validated RMSE | See output above |
| Cross-validated MAE | See output above |

### Interpretation Guide

Each Ridge coefficient represents the change in `log(1 + equity_priority_score)` associated with a **1-standard-deviation increase** in that feature, holding all other features constant.

**Positive coefficients** → feature is associated with **worse equity** (higher priority score).
Example: higher poverty rate → higher equity priority → more transit investment needed.

**Negative coefficients** → feature is associated with **better equity** (lower priority score).
Example: more transit trips → lower equity priority → area is better served.

### Next Steps

This coefficient table provides the *interpretable story* for stakeholders. In Notebook 2, we build an XGBoost model on the same feature set for better predictive accuracy and Sprint 3 simulation capability.